# Pandas Practical Exercises: E-commerce Insights
Six exercises (descriptive statistics, groupby, crosstab/pivot, ranking, window operations, reshaping) on an e-commerce dataset.

**About the data:** no dataset file was provided with the exercises, so a clean e-commerce DataFrame is generated below (300 orders, Jan-Sep 2025, `seed=2026`). If you have the real file, replace the first code cell with `df = pd.read_csv(...)`; the column names (`Order Date`, `Region`, `Category`, `Product`, `Revenue`) are the ones the exercises use.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(seed=2026)
n = 300

prices = {
    "Phone": 300, "Laptop": 800, "Headphones": 60,
    "T-Shirt": 15, "Jeans": 40, "Jacket": 90,
    "Lamp": 25, "Sofa": 350, "Cookware": 70,
    "Novel": 10, "Textbook": 45, "Comics": 8,
}
category_of = {p: c for c, ps in {
    "Electronics": ["Phone", "Laptop", "Headphones"],
    "Clothing": ["T-Shirt", "Jeans", "Jacket"],
    "Home": ["Lamp", "Sofa", "Cookware"],
    "Books": ["Novel", "Textbook", "Comics"],
}.items() for p in ps}

product = rng.choice(list(prices), size=n)
df = pd.DataFrame({
    "Order ID": np.arange(1, n + 1),
    "Order Date": pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 273, size=n), unit="D"),
    "Region": rng.choice(["North", "South", "East", "West"], size=n),
    "Product": product,
    "Quantity": rng.integers(1, 6, size=n),
})
df["Category"] = df["Product"].map(category_of)
df["Unit Price"] = df["Product"].map(prices)
df["Revenue"] = df["Quantity"] * df["Unit Price"]
df = df.sort_values("Order Date").reset_index(drop=True)

print(df.shape)
print(df.isna().sum().sum(), "missing values")
df.head()

(300, 8)
0 missing values


,Order ID,Order Date,Region,Product,Quantity,Category,Unit Price,Revenue
0,85,2025-01-02,West,Phone,2,Electronics,300,600
1,254,2025-01-03,South,Sofa,3,Home,350,1050
2,183,2025-01-03,North,Headphones,2,Electronics,60,120
3,207,2025-01-03,West,Laptop,5,Electronics,800,4000
4,236,2025-01-05,East,Lamp,4,Home,25,100


## Exercise 1: Descriptive Statistics
Total revenue, average order value, and growth percentage.

In [2]:
print(df["Revenue"].describe())
print("Total revenue:       ", df["Revenue"].sum())
print("Average order value: ", round(df["Revenue"].mean(), 2))

# Growth: pct_change needs an ordered series, so use monthly revenue totals
monthly = df.groupby(df["Order Date"].dt.to_period("M"))["Revenue"].sum()
growth = monthly.pct_change() * 100
print(pd.DataFrame({"Revenue": monthly, "Growth %": growth.round(2)}))
print(f"Overall growth, first to last month: {(monthly.iloc[-1] / monthly.iloc[0] - 1) * 100:.2f}%")

count     300.000000
mean      487.516667
std       835.351205
min         8.000000
25%        45.000000
50%       140.000000
75%       450.000000
max      4000.000000
Name: Revenue, dtype: float64
Total revenue:        146255
Average order value:  487.52
            Revenue  Growth %
Order Date                   
2025-01       22716       NaN
2025-02       12101    -46.73
2025-03       19332     59.76
2025-04        7851    -59.39
2025-05       17523    123.19
2025-06       19630     12.02
2025-07       17360    -11.56
2025-08       15689     -9.63
2025-09       14053    -10.43
Overall growth, first to last month: -38.14%


`pct_change()` on raw orders would compare each order with the previous order, which means nothing, so growth is measured on **monthly totals**. Each value is `(this month - previous month) / previous month * 100`; the first month has no previous month, so it is `NaN`.

## Exercise 2: GroupBy & Aggregation

In [3]:
region_cat = df.groupby(["Region", "Category"])["Revenue"].sum().reset_index()
region_cat

,Region,Category,Revenue
0,East,Books,755
1,East,Clothing,3560
2,East,Electronics,24000
3,East,Home,11695
4,North,Books,415
5,North,Clothing,2925
6,North,Electronics,26340
7,North,Home,11805
8,South,Books,1220
9,South,Clothing,2370


`groupby` makes one group per Region/Category pair and `sum()` adds up the revenue in each. The result has the group keys as the index; `reset_index()` turns them back into normal columns, giving a regular DataFrame.

## Exercise 3: Crosstabs & Pivot Tables

In [4]:
print(pd.crosstab(df["Region"], df["Category"]))   # number of orders
pd.pivot_table(df, values="Revenue", index="Region", columns="Category", aggfunc="sum", fill_value=0)

Category  Books  Clothing  Electronics  Home
Region                                      
East         14        21           16    21
North        12        24           20    16
South        25        19           14    23
West         18        22           17    18


Category,Books,Clothing,Electronics,Home
Region,,,,
East,755,3560,24000,11695
North,415,2925,26340,11805
South,1220,2370,9900,12970
West,1455,3435,28420,4990


`crosstab` **counts** orders for each Region/Category pair; `pivot_table` with `aggfunc=sum` **adds revenue** for the same pairs. `fill_value=0` replaces combinations with no orders (otherwise `NaN`).

## Exercise 4: Ranking & Sorting
Top 3 products by revenue in each region.

In [5]:
product_rev = df.groupby(["Region", "Product"], as_index=False)["Revenue"].sum()
product_rev["RevenueRank"] = product_rev.groupby("Region")["Revenue"].rank(method="dense", ascending=False)
top_3_per_region = (product_rev[product_rev["RevenueRank"] <= 3]
                    .sort_values(["Region", "RevenueRank"]))
top_3_per_region

,Region,Product,Revenue,RevenueRank
6,East,Laptop,19200,1.0
9,East,Sofa,9450,2.0
8,East,Phone,3600,3.0
18,North,Laptop,21600,1.0
21,North,Sofa,10850,2.0
20,North,Phone,3900,3.0
33,South,Sofa,11900,1.0
30,South,Laptop,4800,2.0
32,South,Phone,3900,3.0
42,West,Laptop,20800,1.0


Revenue is first summed **per product within each region**, then ranked inside each region (`groupby('Region')` + `rank`). `ascending=False` makes rank 1 the biggest revenue, and `method='dense'` gives consecutive ranks (1, 2, 3, ...) even when there are ties. Filtering `<= 3` keeps the top 3.

## Exercise 5: Window Operations

In [6]:
ts = df.copy()
ts["Order Date"] = pd.to_datetime(ts["Order Date"])
ts = ts.set_index("Order Date")

monthly_rev = ts["Revenue"].resample("ME").sum().to_frame("Revenue")   # monthly totals first (tip)
monthly_rev["rolling_avg"] = monthly_rev["Revenue"].rolling(window=3).mean()
monthly_rev

,Revenue,rolling_avg
Order Date,,
2025-01-31,22716,NaN
2025-02-28,12101,NaN
2025-03-31,19332,18049.666667
2025-04-30,7851,13094.666667
2025-05-31,17523,14902.000000
2025-06-30,19630,15001.333333
2025-07-31,17360,18171.000000
2025-08-31,15689,17559.666667
2025-09-30,14053,15700.666667


A DatetimeIndex lets `resample('ME')` group orders by month-end. `rolling(window=3).mean()` then averages each month with the 2 months before it, smoothing out the ups and downs. The first two months are `NaN` because a full 3-month window does not exist yet.

## Exercise 6: Data Reshaping

In [7]:
# Build a wide table: one row per Region, one column per quarter
df["Quarter"] = "Q" + df["Order Date"].dt.quarter.astype(str) + "_Sales"
df_wide = (df.pivot_table(index="Region", columns="Quarter", values="Revenue", aggfunc="sum")
             .reset_index())
df_wide.columns.name = None
print("Wide:")
print(df_wide)

# Wide to Long
df_long = pd.melt(df_wide, id_vars=["Region"], value_vars=["Q1_Sales", "Q2_Sales", "Q3_Sales"],
                  var_name="Quarter", value_name="Sales")
print("Long:")
print(df_long)

# Long to Wide
df_wide_again = df_long.pivot(index="Region", columns="Quarter", values="Sales")
print("Wide again:")
print(df_wide_again)
print("Round trip matches original:", df_wide_again.equals(df_wide.set_index("Region").rename_axis(columns="Quarter")))

Wide:
  Region  Q1_Sales  Q2_Sales  Q3_Sales
0   East     17930     11046     11034
1  North     12580     13035     15870
2  South      7962      7654     10844
3   West     15677     13269      9354
Long:
   Region   Quarter  Sales
0    East  Q1_Sales  17930
1   North  Q1_Sales  12580
2   South  Q1_Sales   7962
3    West  Q1_Sales  15677
4    East  Q2_Sales  11046
5   North  Q2_Sales  13035
6   South  Q2_Sales   7654
7    West  Q2_Sales  13269
8    East  Q3_Sales  11034
9   North  Q3_Sales  15870
10  South  Q3_Sales  10844
11   West  Q3_Sales   9354
Wide again:
Quarter  Q1_Sales  Q2_Sales  Q3_Sales
Region                               
East        17930     11046     11034
North       12580     13035     15870
South        7962      7654     10844
West        15677     13269      9354
Round trip matches original: True


`melt` turns the quarter columns into rows (one row per Region/Quarter, with the quarter name in `Quarter` and the number in `Sales`). `pivot` reverses it. It needs no aggregation because each Region/Quarter pair appears exactly once.

## Final Summary
Using one clean e-commerce DataFrame I calculated overall revenue statistics and monthly growth, summed revenue by Region and Category, built a crosstab (order counts) and a pivot table (revenue), ranked the top 3 products per region, smoothed monthly revenue with a 3-month rolling average, and converted the data between wide and long format with `melt` and `pivot`.